# FR→EN Transformer: Colab training
Full guide: `docs/COLAB_SETUP.md`.

1. **Runtime → Change runtime type → T4 GPU**
2. Add the secrets `HF_TOKEN` and `WANDB_API_KEY` (key icon in the left bar), with notebook access switched on
3. Run the cells in order. After a disconnect, **run all cells again**: training resumes from the checkpoint on Google Drive.

In [ ]:
# 1. Google Drive + secrets
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
for k in ['HF_TOKEN', 'WANDB_API_KEY']:
    try:
        os.environ[k] = userdata.get(k)
    except Exception as e:
        raise SystemExit(f'Secret {k} missing or notebook access not enabled: {e}')
print('secrets loaded:', [k for k in ['HF_TOKEN', 'WANDB_API_KEY'] if os.environ.get(k)])
OUT = '/content/drive/MyDrive/fr-en-transformer/base'
WORK_CACHE = '/content/drive/MyDrive/fr-en-transformer/work'

In [ ]:
# 2. Code + dependencies + GPU check
%cd /content
!test -d transfromer-model || git clone https://github.com/bad2212/transfromer-model.git
%cd /content/transfromer-model
!git pull -q
!pip -q install sentencepiece wandb datasets safetensors pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Data + tokenizer (~10 min the first time, then cached on Drive so every session uses the same spm.model)
!mkdir -p $WORK_CACHE
!test -f $WORK_CACHE/spm.model || (python -m src.prepare_data --config configs/base.yaml && cp work/* $WORK_CACHE/)
!mkdir -p work && cp -n $WORK_CACHE/* work/
!cat work/data_stats.json

In [ ]:
# 4. Sanity tests + throughput check (~5 min, W&B off). Note tgt_tokens_per_s.
#    < ~12000 tokens/s => use configs/base_6x3.yaml in cells 5 and 6 instead of base.yaml
!python tests/test_sanity.py
!WANDB_MODE=disabled python -m src.train --config configs/base.yaml --out_dir /content/throughput_test --max_steps 300 2>&1 | grep -E 'params|step [0-9]+00 |Error|Traceback'
!rm -rf /content/throughput_test

In [ ]:
# 5. MAIN RUN (~5.5 h budget, resumable). After a disconnect: re-run cells 1-3, then this one.
CONFIG = 'configs/base.yaml'
!python -m src.train --config $CONFIG --out_dir $OUT

In [ ]:
# 6. Checkpoint averaging + dev decoding sweep + test predictions
!python -m src.predict --config $CONFIG --out_dir $OUT --avg 5 --sweep
!python score.py --gold data/dev/labels.jsonl --pred $OUT/eval/dev_predictions.json
!cp $OUT/eval/test_predictions.json $OUT/../test_predictions.json && ls -la $OUT/eval